# Path Fix & Imports

In [1]:
import sys
import os

# 1. Fix the path so Jupyter can find your custom modules
# This goes up one level from /notebooks to the root project folder
root_dir = os.path.abspath('..')
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

# 2. PySpark Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 3. Custom ETL Imports
from etl_pipeline.utils.spark_session import get_spark_session
from etl_pipeline.utils.s3_reader import read_delta_table
from etl_pipeline.utils.s3_writer import write_delta_table

print("✅ Paths configured and modules imported successfully!")

✅ Paths configured and modules imported successfully!


# Start the Spark Session

In [2]:
# Initialize Spark for the Gold Layer
spark = get_spark_session("GoldLayer-Testing")

print("✅ Spark Session is running!")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bbf2e768-456f-4802-a625-b7439003d102;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.2/delta-spark_2.12-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.2!delta-spark_2.12.jar (837ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.3.2!delta-storage.jar (153ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (209ms)
:: resolution report :: resolve 4922ms :: artifacts dl 

✅ Spark Session is running!


# dim_products & dim_sellers

In [5]:
print("Loading Silver tables...")

# For dim_products
df_silver_products = read_delta_table(spark, "silver", "silver_olist_products_dataset")
df_silver_translation = read_delta_table(spark, "silver", "silver_product_category_name_translation")

# For dim_sellers (and later dim_customers)
df_silver_sellers = read_delta_table(spark, "silver", "silver_olist_sellers_dataset")
df_silver_geo = read_delta_table(spark, "silver", "silver_olist_geolocation_dataset")

print("✅ Silver tables loaded successfully!")

# Let's just peek at the sellers to make sure it worked
df_silver_sellers.show(5)

Loading Silver tables...


26/04/10 02:19:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


✅ Silver tables loaded successfully!


26/04/10 02:19:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 8:>                                                          (0 + 1) / 1]

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                 04195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows



# dim_products

In [7]:
# --- Build dim_products ---
print("Transforming dim_products...")

# 1. Join the products table with the translation table
df_dim_products = df_silver_products.join(
    df_silver_translation,
    on="product_category_name",
    how="left"
)

# 2. Select and rename the columns for the final Gold dimension
df_dim_products = df_dim_products.select(
    F.col("product_id"),
    F.col("product_category_name").alias("product_category_pt"), # Keep Portuguese
    F.col("product_category_name_english").alias("product_category_en"), # Keep English
    F.col("product_weight_g"),
    F.col("product_length_cm"),
    F.col("product_height_cm"),
    F.col("product_width_cm")
)

# 3. Verify the transformation
df_dim_products.show(5)
print(f"Total rows in dim_products: {df_dim_products.count()}")

Transforming dim_products...


+--------------------+--------------------+-------------------+----------------+-----------------+-----------------+----------------+
|          product_id| product_category_pt|product_category_en|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+--------------------+-------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|          perfumaria|          perfumery|             225|               16|               10|              14|
|3aa071139cb16b67c...|               artes|                art|            1000|               30|               18|              20|
|96bd76ec8810374ed...|       esporte lazer|     sports leisure|             154|               18|                9|              15|
|cef67bcfe19066a93...|               bebes|               baby|             371|               26|                4|              26|
|9dc1a7de274444849...|utilidades domest...|         housewares

[Stage 41:>                                                         (0 + 1) / 1]

Total rows in dim_products: 32951


# dim_sellers

In [8]:
# --- Build dim_sellers ---
print("Transforming dim_sellers...")

# 1. Join sellers with geolocation to get their exact latitude/longitude
# We use a LEFT join because some sellers might have invalid/missing zip codes
df_dim_sellers = df_silver_sellers.join(
    df_silver_geo,
    df_silver_sellers["seller_zip_code_prefix"] == df_silver_geo["geolocation_zip_code_prefix"],
    how="left"
)

# 2. Select and rename the columns for the final Gold dimension
df_dim_sellers = df_dim_sellers.select(
    F.col("seller_id"),
    F.col("seller_city"),
    F.col("seller_state"),
    F.col("seller_zip_code_prefix"),
    F.col("geolocation_lat").alias("seller_lat"),
    F.col("geolocation_lng").alias("seller_lng")
)

# 3. Verify the transformation
df_dim_sellers.show(5)
print(f"Total rows in dim_sellers: {df_dim_sellers.count()}")

Transforming dim_sellers...


+--------------------+-----------+------------+----------------------+-------------------+-------------------+
|           seller_id|seller_city|seller_state|seller_zip_code_prefix|         seller_lat|         seller_lng|
+--------------------+-----------+------------+----------------------+-------------------+-------------------+
|3442f8959a84dea7e...|   campinas|          SP|                 13023| -22.89031793005942|  -47.0610973760748|
|3442f8959a84dea7e...|   campinas|          SP|                 13023|-22.891019217871794| -47.06036512254021|
|3442f8959a84dea7e...|   campinas|          SP|                 13023|-22.896153837642057|-47.062431210369965|
|3442f8959a84dea7e...|   campinas|          SP|                 13023|-22.898721582578283| -47.06294687552652|
|3442f8959a84dea7e...|   campinas|          SP|                 13023|-22.884841886202345|   -47.045900755208|
+--------------------+-----------+------------+----------------------+-------------------+-------------------+
o

[Stage 56:>                                                         (0 + 1) / 1]

Total rows in dim_sellers: 303892


In [10]:
# --- Build dim_sellers (CORRECTED) ---
print("Transforming dim_sellers...")

# 1. Create a unique Geolocation table (1 row per zip code) by averaging the coordinates
df_geo_centroid = df_silver_geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng")
)

# 2. Join sellers with the NEW centroid table
df_dim_sellers = df_silver_sellers.join(
    df_geo_centroid,
    df_silver_sellers["seller_zip_code_prefix"] == df_geo_centroid["geolocation_zip_code_prefix"],
    how="left"
)

# 3. Select and rename the columns for the final Gold dimension
df_dim_sellers = df_dim_sellers.select(
    F.col("seller_id"),
    F.col("seller_city"),
    F.col("seller_state"),
    F.col("seller_zip_code_prefix"),
    F.col("geolocation_lat").alias("seller_lat"),
    F.col("geolocation_lng").alias("seller_lng")
)

# 4. Verify the transformation
df_dim_sellers.show(5)
print(f"Total rows in dim_sellers: {df_dim_sellers.count()}")

Transforming dim_sellers...


+--------------------+-----------------+------------+----------------------+-------------------+-------------------+
|           seller_id|      seller_city|seller_state|seller_zip_code_prefix|         seller_lat|         seller_lng|
+--------------------+-----------------+------------+----------------------+-------------------+-------------------+
|3442f8959a84dea7e...|         campinas|          SP|                 13023|-22.893317190455804|-47.060596205366885|
|d1b65fc7debc3361e...|       mogi guacu|          SP|                 13844| -22.38305943825609| -46.94803398002504|
|ce3ad9de960102d06...|   rio de janeiro|          RJ|                 20031| -22.90944638719964| -43.18023952743163|
|c0f3eea2e14555b6f...|        sao paulo|          SP|                 04195|-23.657064435722678| -46.61266825096129|
|51a04a8a6bdcb23de...|braganca paulista|          SP|                 12914|-22.964518123035013| -46.53425667204075|
+--------------------+-----------------+------------+-----------

# dim_customers

In [11]:
print("Loading Silver Customers...")
df_silver_customers = read_delta_table(spark, "silver", "silver_olist_customers_dataset")

# --- Build dim_customers ---
print("Transforming dim_customers...")

# 1. We re-create the centroid table just in case you run this cell independently later
df_geo_centroid = df_silver_geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng")
)

# 2. Join customers with the centroid table (LEFT join for the same reason as sellers)
df_dim_customers = df_silver_customers.join(
    df_geo_centroid,
    df_silver_customers["customer_zip_code_prefix"] == df_geo_centroid["geolocation_zip_code_prefix"],
    how="left"
)

# 3. Select and rename the columns for the final Gold dimension
df_dim_customers = df_dim_customers.select(
    F.col("customer_id"),
    F.col("customer_unique_id"),
    F.col("customer_city"),
    F.col("customer_state"),
    F.col("customer_zip_code_prefix"),
    F.col("geolocation_lat").alias("customer_lat"),
    F.col("geolocation_lng").alias("customer_lng")
)

# 4. Verify the transformation
df_dim_customers.show(5)
print(f"Total rows in dim_customers: {df_dim_customers.count()}")

Loading Silver Customers...
Transforming dim_customers...


+--------------------+--------------------+--------------------+--------------+------------------------+-------------------+-------------------+
|         customer_id|  customer_unique_id|       customer_city|customer_state|customer_zip_code_prefix|       customer_lat|       customer_lng|
+--------------------+--------------------+--------------------+--------------+------------------------+-------------------+-------------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|             jacunda|            PA|                   68590| -4.450626103818663|-49.115538072421536|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|sao jose do rio p...|            SP|                   15056| -20.77555084682476| -49.33648682199814|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                 itu|            SP|                   13302|-23.265013304778247|-47.274141222587915|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|             coaraci|            BA|                   45638|-14.635468320077546|-39.55

# dim_date

In [12]:
# --- Build dim_date ---
print("Generating dim_date...")

# 1. Define the start and end dates for the Olist dataset
start_date = "2016-01-01"
end_date = "2018-12-31"

# 2. Generate a dataframe with a single row for every day in that range
df_dim_date = spark.sql(f"""
    SELECT sequence(
        to_date('{start_date}'), 
        to_date('{end_date}'), 
        interval 1 day
    ) as date_array
""").withColumn("full_date", F.explode("date_array")).drop("date_array")

# 3. Extract the useful date parts for analytics
df_dim_date = df_dim_date.select(
    F.date_format(F.col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
    F.col("full_date"),
    F.dayofmonth(F.col("full_date")).alias("day_of_month"),
    F.dayofweek(F.col("full_date")).alias("day_of_week"), # 1 = Sunday, 7 = Saturday
    F.date_format(F.col("full_date"), "EEEE").alias("day_name"),
    F.month(F.col("full_date")).alias("month"),
    F.quarter(F.col("full_date")).alias("quarter"),
    F.year(F.col("full_date")).alias("year")
)

# 4. Add a helpful boolean flag for weekends (Saturday or Sunday)
df_dim_date = df_dim_date.withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False)
)

# 5. Verify the transformation
df_dim_date.show(10)
print(f"Total rows in dim_date: {df_dim_date.count()}")

Generating dim_date...
+--------+----------+------------+-----------+---------+-----+-------+----+----------+
|date_key| full_date|day_of_month|day_of_week| day_name|month|quarter|year|is_weekend|
+--------+----------+------------+-----------+---------+-----+-------+----+----------+
|20160101|2016-01-01|           1|          6|   Friday|    1|      1|2016|     false|
|20160102|2016-01-02|           2|          7| Saturday|    1|      1|2016|      true|
|20160103|2016-01-03|           3|          1|   Sunday|    1|      1|2016|      true|
|20160104|2016-01-04|           4|          2|   Monday|    1|      1|2016|     false|
|20160105|2016-01-05|           5|          3|  Tuesday|    1|      1|2016|     false|
|20160106|2016-01-06|           6|          4|Wednesday|    1|      1|2016|     false|
|20160107|2016-01-07|           7|          5| Thursday|    1|      1|2016|     false|
|20160108|2016-01-08|           8|          6|   Friday|    1|      1|2016|     false|
|20160109|2016-01-09

# fact_sales

In [4]:
# --- Build fact_sales ---
print("Loading Silver tables for fact_sales...")
df_silver_orders = read_delta_table(spark, "silver", "silver_olist_orders_dataset")
df_silver_items = read_delta_table(spark, "silver", "silver_olist_order_items_dataset")
df_silver_payments = read_delta_table(spark, "silver", "silver_olist_order_payments_dataset")

print("Transforming fact_sales...")

# 1. Join Order Items with Orders to get the purchase timestamp and customer ID
df_sales_base = df_silver_items.join(
    df_silver_orders,
    on="order_id",
    how="inner"
)

# 2. Aggregate payments per order (some orders have multiple payments, e.g., split across two credit cards)
# We sum the payment value to avoid duplicating order items during the join
df_payments_agg = df_silver_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value")
)

# 3. Join the aggregated payments to our base sales table
df_sales_joined = df_sales_base.join(
    df_payments_agg,
    on="order_id",
    how="left"
)

# 4. Select and calculate the final metrics and foreign keys (with rounding!)
df_fact_sales = df_sales_joined.select(
    F.col("order_id"),
    F.col("order_item_id"),
    F.col("product_id"),
    F.col("seller_id"),
    F.col("customer_id"),
    F.col("order_status"),
    F.date_format(F.col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("purchase_date_key"),
    
    # Round all monetary values to 2 decimal places
    F.round(F.col("price"), 2).alias("price"),
    F.round(F.col("freight_value"), 2).alias("freight_value"),
    F.round(F.col("price") + F.col("freight_value"), 2).alias("total_item_value"),
    F.round(F.col("total_payment_value"), 2).alias("total_payment_value")
)

# 5. Verify the transformation
df_fact_sales.show(5)
print(f"Total rows in fact_sales: {df_fact_sales.count()}")

Loading Silver tables for fact_sales...


26/04/10 06:18:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Transforming fact_sales...


26/04/10 06:18:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------+-------------+--------------------+--------------------+--------------------+------------+-----------------+-----+-------------+----------------+-------------------+
|            order_id|order_item_id|          product_id|           seller_id|         customer_id|order_status|purchase_date_key|price|freight_value|total_item_value|total_payment_value|
+--------------------+-------------+--------------------+--------------------+--------------------+------------+-----------------+-----+-------------+----------------+-------------------+
|995392413cee61cc1...|            1|2e58d459f2a674ee7...|a888faf2d1baececa...|4bf24904ec428325a...|   delivered|         20170904|129.9|        13.93|          143.83|             143.83|
|b39de9ed2bb8fd08a...|            1|0d85c435fd60b277f...|00fc707aaaad2d313...|ed18b557140ff674f...|   delivered|         20180327| 98.9|         22.4|           121.3|              121.3|
|b1a88554eb1f7f686...|            1|364e789259da982f5...|190

[Stage 19:==================================>                       (3 + 2) / 5]

Total rows in fact_sales: 112650


In [8]:
import pyspark.sql.functions as F

# Show orders where the item value does NOT equal the total payment
df_fact_sales.filter(F.col("total_item_value") != F.col("total_payment_value")).show(8)

[Stage 65:>                                                         (0 + 1) / 1]

+--------------------+-------------+--------------------+--------------------+--------------------+------------+-----------------+-----+-------------+----------------+-------------------+
|            order_id|order_item_id|          product_id|           seller_id|         customer_id|order_status|purchase_date_key|price|freight_value|total_item_value|total_payment_value|
+--------------------+-------------+--------------------+--------------------+--------------------+------------+-----------------+-----+-------------+----------------+-------------------+
|750ed62dff96bf571...|            2|4b77167175713bd04...|d6cd01c59123df02f...|a05f88744e7de808e...|    canceled|         20180528| 87.5|         7.48|           94.98|             189.96|
|750ed62dff96bf571...|            1|4b77167175713bd04...|d6cd01c59123df02f...|a05f88744e7de808e...|    canceled|         20180528| 87.5|         7.48|           94.98|             189.96|
|3902c863a4cc7b325...|            2|aa280035c50ba62c7...|fa4

# fact_order_fulfillment

In [3]:
# --- Build fact_order_fulfillment ---
print("Loading Silver tables for fact_order_fulfillment...")
df_silver_orders = read_delta_table(spark, "silver", "silver_olist_orders_dataset")
df_silver_reviews = read_delta_table(spark, "silver", "silver_olist_order_reviews_dataset")

print("Transforming fact_order_fulfillment...")

# 1. Join Orders with Reviews
# We use a LEFT join because an order might not have a review yet!
df_fulfillment_base = df_silver_orders.join(
    df_silver_reviews,
    on="order_id",
    how="left"
)

# 2. Select columns, generate Date Keys, and calculate performance metrics
df_fact_fulfillment = df_fulfillment_base.select(
    F.col("order_id"),
    F.col("customer_id"),
    F.col("review_id"),
    F.col("order_status"),
    F.col("review_score"),
    
    # Generate integer Date Keys for the BI tool
    F.date_format(F.col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("purchase_date_key"),
    F.date_format(F.col("order_estimated_delivery_date"), "yyyyMMdd").cast("int").alias("estimated_delivery_date_key"),
    F.date_format(F.col("order_delivered_customer_date"), "yyyyMMdd").cast("int").alias("actual_delivery_date_key"),

    # CALCULATED METRIC 1: Delivery Delay in Days
    F.datediff(
        F.col("order_delivered_customer_date"), 
        F.col("order_estimated_delivery_date")
    ).alias("delivery_delay_days"),
    
    # CALCULATED METRIC 2: Total Shipping Duration in Days
    F.datediff(
        F.col("order_delivered_customer_date"), 
        F.col("order_purchase_timestamp")
    ).alias("shipping_duration_days")
)

# 3. Add a BI-friendly Flag for Late Deliveries (1 = Late, 0 = On Time/Early)
df_fact_fulfillment = df_fact_fulfillment.withColumn(
    "is_late_delivery",
    F.when(F.col("delivery_delay_days") > 0, 1).otherwise(0)
)

# 4. Verify the transformation
df_fact_fulfillment.show(5)
print(f"Total rows in fact_order_fulfillment: {df_fact_fulfillment.count()}")

Loading Silver tables for fact_order_fulfillment...


26/04/10 11:33:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Transforming fact_order_fulfillment...


26/04/10 11:34:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------------+--------------------+--------------------+------------+------------+-----------------+---------------------------+------------------------+-------------------+----------------------+----------------+
|            order_id|         customer_id|           review_id|order_status|review_score|purchase_date_key|estimated_delivery_date_key|actual_delivery_date_key|delivery_delay_days|shipping_duration_days|is_late_delivery|
+--------------------+--------------------+--------------------+------------+------------+-----------------+---------------------------+------------------------+-------------------+----------------------+----------------+
|995392413cee61cc1...|4bf24904ec428325a...|4eb2e2f78619c034e...|   delivered|           4|         20170904|                   20170927|                20170922|                 -5|                    18|               0|
|b39de9ed2bb8fd08a...|ed18b557140ff674f...|1d359fdbfd59b2b84...|   delivered|           5|         20180327|    

[Stage 18:==============================================>           (4 + 1) / 5]

Total rows in fact_order_fulfillment: 99714
